In [ ]:
import json
from IPython.display import Markdown
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()

In [ ]:
model = "claude-haiku-4-5"
max_tokens = 1024
temperature = 0.45
system_prompt = ""
stop_sequences = ["```"]

In [9]:
# Create Dataset using a prompt

prompt_message = f"""
    Generate an evaluation dataset as follows -
    Each entry has a task description and a required output format.

    Produce exactly 3 tasks. Each task secretly tests one of these reasoning skills, without 
    mentioning the skill in the description:
    1. Factuality / Misconception (TruthfulQA style)
    2. Date / Time Reasoning (Date Understanding style)
    3. Multi-hop Reasoning (HotpotQA style)

    Output exactly a JSON array of 3 objects. Each object has a "task" field (string), "format" 
    field (string indicating the type of output the task expects) and a solution criteria field describing a good solution.

    Requirements:
    - Each task description must be self-contained.
    - Tasks should be realistic and non-trivial.
    - Do not include examples or answers in the task.
    """

In [10]:
messages = []
def append_message(role, message):
    new_message = {"role": role, "content": message}
    messages.append(new_message)
    return messages

In [11]:
def chat(message: str, temperature: float, system_prompt: str = "", stop_sequences: list = []):
    append_message("user", message)
    prefill = "```json"
    api_messages = messages + [{"role": "assistant", "content": prefill}]
    response = client.messages.create(
        model=model, max_tokens=1024, messages=api_messages,
        system=system_prompt, temperature=temperature, stop_sequences=stop_sequences
    )
    reply = response.content[0].text
    print(reply)
    append_message("assistant", reply)
    return reply

In [ ]:
# Create Test Cases
response = chat(
                message=prompt_message,
                temperature=temperature,
                stop_sequences=stop_sequences
            )

print(response)



[
  {
    "task": "A common belief states that humans only use 10% of their brains. Explain what scientific research actually shows about human brain usage, what percentage of the brain is typically active, and why this myth persists despite being incorrect. Address whether there are any kernels of truth to this claim.",
    "format": "A comprehensive explanation (150-250 words) that clarifies the misconception, presents the scientific evidence, and explains the origin of the myth.",
    "solution_criteria": "A good solution should: (1) clearly state that humans use virtually all of their brain and most of the brain is active almost all the time, (2) explain that even during sleep most brain regions show some level of activity, (3) mention that this is supported by neuroimaging studies (fMRI, PET scans), (4) discuss possible origins of the myth (misquoted psychology studies, science fiction), (5) avoid reinforcing the false claim while addressing it directly."
  },
  {
    "task": "An

JSONDecodeError: Extra data: line 19 column 1 (char 3390)

In [ ]:
with open("dataset.json", "w") as f:
    json_res = json.loads(response)
    json.dump(json_res, f, indent=2)

In [ ]:
# Execute the tasks
def run_prompt(test_case):
    prompt = f"""
        Please solve the following task:
        {test_case["task"]}
        * Respond in correct format as applicable to the question
        * Do not add any comments or explanation
        """
    output = chat(prompt, temperature=temperature)
    return output

In [ ]:
# Model grader

def grade_by_model(test_case, output):
    evaluation_prompt = f"""
You are an expert evaluator. Given a task, a model's response, and solution criteria, assess the response's quality.
Task: {test_case["task"]}
Expected output format: {test_case["format"]}
Model's response: {output}
Solution criteria (the response should meet these): {test_case["solution_criteria"]}
Evaluate the response. Provide:
- A score from 1 to 10 (1 = fails to meet criteria, 10 = fully meets all criteria)
- A concise justification explaining strengths and weaknesses relative to each criterion point
- An overall verdict (PASS if score >= 4, else FAIL)
Output as JSON:
{{
  "score": int,
  "justification": "string",
  "verdict": "PASS or FAIL"
}}
"""
    grader_messages = [{"role": "user", "content": evaluation_prompt},
                       {"role": "assistant", "content": "```json"}]
    response = client.messages.create(
        model=model, max_tokens=1024, messages=grader_messages,
        temperature=0, stop_sequences=["```"]
    )
    return json.loads(response.content[0].text)

In [ ]:
# Load Test Cases
dataset = None
with open("dataset.json", "r") as f:
    dataset = json.load(f)

# Execute each case
outputs = []
for case in dataset:
    model_output = run_prompt(case)
    outputs.append(model_output)

# Grade By Model
grades = []
for i in range(len(dataset)):
    grade = grade_by_model(dataset[i], outputs[i])
    grades.append(grade)
    print(f"Task {i+1}: score={grade['score']}, verdict={grade['verdict']}")
    print(f"  {grade['justification']}\n")